# Simple request -> response -> parsing flow

This notebook shows the basic scraping loop using an eBay search page as an example:

1. Build a search URL.
2. Send an HTTP request with headers.
3. Check the response.
4. Parse HTML with BeautifulSoup.
5. Store parsed rows in a pandas DataFrame.


In [19]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlencode


## 1. Build the request URL

Keep the search term and request parameters in variables so the flow is easy to reuse.


In [20]:
search_term = "laptop"
base_url = "https://www.ebay.com/sch/i.html"

params = {
    "_nkw": search_term,
    "_sacat": 0,
}

url = f"{base_url}?{urlencode(params)}"
url


'https://www.ebay.com/sch/i.html?_nkw=laptop&_sacat=0'

## 2. Send the request

A browser-like `User-Agent` helps the server understand this is a normal HTML page request.


In [21]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers=headers, timeout=20)

print("Status code:", response.status_code)
print("Final URL:", response.url)
print("Content type:", response.headers.get("content-type"))
print("HTML characters:", len(response.text))

response.raise_for_status()
html = response.text


Status code: 403
Final URL: https://www.ebay.com/sch/i.html?_nkw=laptop&_sacat=0
Content type: text/html
HTML characters: 1832


HTTPError: 403 Client Error: Forbidden for url: https://www.ebay.com/sch/i.html?_nkw=laptop&_sacat=0

## 3. Parse the HTML response

BeautifulSoup turns the raw HTML string into a searchable document tree.


In [22]:
soup = BeautifulSoup(html, "html.parser")

page_title = soup.title.get_text(strip=True) if soup.title else None
cards = soup.select("li.s-item")

print("Page title:", page_title)
print("Product cards found:", len(cards))


Page title: Error Page | eBay
Product cards found: 0


## 4. Extract fields from each product card

Small helper functions keep the parsing code readable and avoid errors when an element is missing.


In [ ]:
def text_or_none(parent, selector):
    element = parent.select_one(selector)
    if element is None:
        return None
    return element.get_text(" ", strip=True)


def attr_or_none(parent, selector, attr):
    element = parent.select_one(selector)
    if element is None:
        return None
    return element.get(attr)


def parse_product(card):
    title = text_or_none(card, ".s-item__title")

    # eBay often includes a placeholder card; skip it.
    if not title or title.lower() == "shop on ebay":
        return None

    return {
        "title": title,
        "price": text_or_none(card, ".s-item__price"),
        "shipping": text_or_none(card, ".s-item__shipping, .s-item__logisticsCost"),
        "location": text_or_none(card, ".s-item__location"),
        "seller": text_or_none(card, ".s-item__seller-info-text"),
        "link": attr_or_none(card, ".s-item__link", "href"),
    }


products = []
for card in cards:
    product = parse_product(card)
    if product:
        products.append(product)

len(products)


## 5. Convert parsed rows into a DataFrame

A DataFrame makes it easy to preview, filter, clean, and export the scraped data.


In [ ]:
df = pd.DataFrame(products)
df.head(10)


## 6. Optional: save the parsed data

Uncomment this cell when you want to export the results to CSV.


In [ ]:
# output_path = "ebay_laptop_results.csv"
# df.to_csv(output_path, index=False)
# print(f"Saved {len(df)} rows to {output_path}")
